# Matrix Rank - Examples

Interactive demonstrations of matrix rank computation and applications in machine learning.

**Topics Covered:**
1. Basic rank computation
2. Rank via SVD
3. Effects of rank deficiency
4. Rank-nullity theorem
5. Row echelon form method
6. Feature redundancy in ML
7. Low-rank approximation
8. Matrix completion (recommender systems)
9. Numerical rank issues
10. Rank under operations

In [ ]:
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

---
## 1. Basic Rank Computation

The **rank** of a matrix is the maximum number of linearly independent rows (or columns).

$$\text{rank}(A) = \dim(\text{column space}) = \dim(\text{row space})$$

In [ ]:
print("="*60)
print("EXAMPLE 1: Basic Rank Computation")
print("="*60)

# Full rank matrix (2×2, columns are independent)
A = np.array([[1, 2],
              [3, 4]])
print(f"Matrix A:\n{A}")
print(f"rank(A) = {np.linalg.matrix_rank(A)}")
print("Full rank: columns are linearly independent")

print("\n" + "-"*40)

# Rank-deficient matrix (row 2 = 2 × row 1)
B = np.array([[1, 2],
              [2, 4]])
print(f"Matrix B:\n{B}")
print(f"rank(B) = {np.linalg.matrix_rank(B)}")
print("Rank-deficient: Row 2 = 2 × Row 1")

print("\n" + "-"*40)

# Rectangular matrix
C = np.array([[1, 2, 3],
              [4, 5, 6]])
print(f"Matrix C (2×3):\n{C}")
print(f"rank(C) = {np.linalg.matrix_rank(C)}")
print(f"Upper bound: min(2, 3) = {min(C.shape)}")

print("\n" + "-"*40)

# Special cases
print(f"Zero matrix (3×3): rank = {np.linalg.matrix_rank(np.zeros((3,3)))}")
print(f"Identity (4×4): rank = {np.linalg.matrix_rank(np.eye(4))}")

---
## 2. Rank via SVD (Singular Value Decomposition)

The rank equals the number of **non-zero singular values**.

$$A = U\Sigma V^T$$
$$\text{rank}(A) = \#\{\sigma_i > 0\}$$

In [ ]:
print("="*60)
print("EXAMPLE 2: Rank via SVD")
print("="*60)

# Create a rank-2 matrix (3×4)
A = np.array([[1, 2, 3, 4],
              [2, 4, 6, 8],   # 2 × Row 1
              [1, 0, 1, 2]])

print(f"Matrix A (3×4):\n{A}")
print("\nNote: Row 2 = 2 × Row 1 (dependent!)")

# SVD
U, S, Vt = np.linalg.svd(A)

print(f"\nSingular values: {S}")
print(f"Non-zero singular values (>1e-10): {np.sum(S > 1e-10)}")
print(f"rank(A) = {np.linalg.matrix_rank(A)}")

# Visualization
plt.figure(figsize=(10, 4))

plt.subplot(121)
plt.bar(range(len(S)), S, color='steelblue')
plt.axhline(1e-10, color='r', linestyle='--', label='Zero threshold')
plt.xlabel('Singular Value Index')
plt.ylabel('Value')
plt.title('Singular Values of Rank-2 Matrix')
plt.legend()

# Full rank example
B = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 10]])  # Last entry differs to be full rank

_, S_B, _ = np.linalg.svd(B)

plt.subplot(122)
plt.bar(range(len(S_B)), S_B, color='green')
plt.xlabel('Singular Value Index')
plt.ylabel('Value')
plt.title(f'Singular Values of Full Rank Matrix (rank={np.linalg.matrix_rank(B)})')

plt.tight_layout()
plt.show()

---
## 3. Effects of Rank Deficiency

Rank determines whether $Ax = b$ has solutions:
- **Full rank** → unique solution
- **Rank deficient + consistent** → infinitely many solutions
- **Rank deficient + inconsistent** → no solution

In [ ]:
print("="*60)
print("EXAMPLE 3: Effects of Rank Deficiency")
print("="*60)

# Full rank system: unique solution
print("CASE 1: Full Rank System")
print("-"*40)
A_full = np.array([[1, 2],
                   [3, 4]])
b = np.array([5, 11])

print(f"A = {A_full.tolist()}")
print(f"b = {b.tolist()}")
print(f"rank(A) = {np.linalg.matrix_rank(A_full)}")

x = np.linalg.solve(A_full, b)
print(f"Unique solution: x = {x}")
print(f"Verification: Ax = {A_full @ x}")

# Rank-deficient, consistent case
print("\nCASE 2: Rank Deficient, Consistent")
print("-"*40)
A_def = np.array([[1, 2],
                  [2, 4]])  # Row 2 = 2×Row 1
b1 = np.array([3, 6])       # Consistent (6 = 2×3)

print(f"A = {A_def.tolist()}")
print(f"b = {b1.tolist()}")
print(f"rank(A) = {np.linalg.matrix_rank(A_def)}")
print(f"rank([A|b]) = {np.linalg.matrix_rank(np.column_stack([A_def, b1]))}")

x_lstsq = np.linalg.lstsq(A_def, b1, rcond=None)[0]
print(f"Least-norm solution: x = {x_lstsq}")
print("→ Infinitely many solutions exist along the null space!")

# Rank-deficient, inconsistent case
print("\nCASE 3: Rank Deficient, Inconsistent")
print("-"*40)
b2 = np.array([3, 7])  # Inconsistent (7 ≠ 2×3)

print(f"A = {A_def.tolist()}")
print(f"b = {b2.tolist()}")
print(f"rank(A) = {np.linalg.matrix_rank(A_def)}")
print(f"rank([A|b]) = {np.linalg.matrix_rank(np.column_stack([A_def, b2]))}")
print("rank(A) < rank([A|b]) → NO EXACT SOLUTION EXISTS")

---
## 4. Rank-Nullity Theorem

For matrix $A \in \mathbb{R}^{m \times n}$:

$$\boxed{\text{rank}(A) + \text{nullity}(A) = n}$$

- **rank** = dimension of column space (# pivot columns)
- **nullity** = dimension of null space (# free columns)

In [ ]:
print("="*60)
print("EXAMPLE 4: Rank-Nullity Theorem")
print("="*60)

# Create a 3×5 matrix with rank 2
A = np.array([[1, 2, 1, 0, 1],
              [2, 4, 3, 1, 3],
              [3, 6, 4, 1, 4]])

print(f"Matrix A (3×5):\n{A}")

n = A.shape[1]  # Number of columns
rank = np.linalg.matrix_rank(A)
nullity = n - rank

print(f"\nn (columns) = {n}")
print(f"rank(A) = {rank}")
print(f"nullity(A) = n - rank = {n} - {rank} = {nullity}")
print(f"\n✓ Rank-Nullity: {rank} + {nullity} = {rank + nullity} = {n}")

# Find null space basis
null_space = linalg.null_space(A)
print(f"\nNull space dimension: {null_space.shape[1]}")
print(f"Null space basis (each column is a basis vector):\n{null_space}")

# Verify null space vectors
print("\nVerification: A × (null space vectors) = 0")
for i in range(null_space.shape[1]):
    v = null_space[:, i]
    Av = A @ v
    print(f"  A × v_{i+1} = {Av} ≈ 0 ✓")

---
## 5. Rank via Row Echelon Form

Rank = **number of pivots** (non-zero rows in row echelon form).

In [ ]:
print("="*60)
print("EXAMPLE 5: Rank via Row Echelon Form")
print("="*60)

A = np.array([[1, 2, 3, 4],
              [2, 4, 7, 9],
              [3, 6, 10, 13]], dtype=float)

print(f"Original matrix A:\n{A}")

# Manual row reduction
U = A.copy()

# Step 1: Eliminate column 1
U[1] = U[1] - 2 * U[0]
U[2] = U[2] - 3 * U[0]
print(f"\nAfter R2 ← R2 - 2R1, R3 ← R3 - 3R1:\n{U}")

# Step 2: Eliminate below second pivot
U[2] = U[2] - U[1]
print(f"\nAfter R3 ← R3 - R1:\n{U}")

# Count non-zero rows
non_zero_rows = np.sum(~np.allclose(U, 0, axis=1))
print(f"\nNumber of non-zero rows (pivots): {non_zero_rows}")
print(f"Pivot columns: 1 and 3")
print(f"\nrank(A) via NumPy: {np.linalg.matrix_rank(A)} ✓")

---
## 6. Feature Redundancy in Machine Learning

If rank(X) < number of features, some features are redundant (linear combinations of others).

In [ ]:
print("="*60)
print("EXAMPLE 6: Feature Redundancy in ML")
print("="*60)

np.random.seed(42)
n_samples = 100

# Create features with redundancy
x1 = np.random.randn(n_samples)
x2 = np.random.randn(n_samples)
x3 = 2 * x1 + 3 * x2  # REDUNDANT: linear combination!
x4 = np.random.randn(n_samples)

X = np.column_stack([x1, x2, x3, x4])

print(f"Data matrix X: {X.shape} (100 samples × 4 features)")
print(f"\nFeature x3 = 2×x1 + 3×x2 (redundant!)")
print(f"\nrank(X) = {np.linalg.matrix_rank(X)}")
print(f"Expected rank: 3 (one redundant feature)")

# Gram matrix X^T X
XtX = X.T @ X
print(f"\nGram matrix XᵀX:")
print(f"rank(XᵀX) = {np.linalg.matrix_rank(XtX)}")
print(f"Condition number: {np.linalg.cond(XtX):.2e}")
print("(Large condition number indicates near-singularity)")

# Singular values reveal redundancy
_, S, _ = np.linalg.svd(X)
print(f"\nSingular values of X: {S}")
print("Note: One singular value is nearly zero!")

# Impact on linear regression
print("\n" + "-"*40)
print("IMPACT ON LINEAR REGRESSION:")
y = x1 + x2 + x4 + 0.1 * np.random.randn(n_samples)

try:
    w = np.linalg.solve(XtX, X.T @ y)
    print(f"Direct solution w = (XᵀX)⁻¹Xᵀy: {w}")
except np.linalg.LinAlgError:
    print("Direct solution failed! (XᵀX is singular)")

# Use pseudo-inverse
w_lstsq = np.linalg.lstsq(X, y, rcond=None)[0]
print(f"Least squares solution (pseudo-inverse): {w_lstsq}")
print("\n→ Solution exists but w[2] depends on arbitrary choice!")

---
## 7. Low-Rank Approximation

The **Eckart-Young theorem** states that the best rank-$k$ approximation is given by truncated SVD:

$$A_k = \sum_{i=1}^k \sigma_i u_i v_i^T$$

In [ ]:
print("="*60)
print("EXAMPLE 7: Low-Rank Approximation")
print("="*60)

np.random.seed(42)

# Create a full-rank matrix
A = np.random.randn(10, 8)

print(f"Original matrix A: {A.shape}")
print(f"Full rank: {np.linalg.matrix_rank(A)}")

# SVD
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print(f"\nSingular values: {np.round(S, 3)}")

# Rank-k approximations
print("\nLow-Rank Approximations:")
print("-" * 50)

results = []
for k in [1, 2, 4, 6, 8]:
    # Truncated SVD
    A_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    
    error = np.linalg.norm(A - A_k, 'fro')
    relative_error = error / np.linalg.norm(A, 'fro')
    compression = A.size / (k * (A.shape[0] + A.shape[1]))
    
    results.append((k, relative_error, compression))
    
    print(f"Rank-{k}: Relative error = {relative_error:.4f}, Compression = {compression:.2f}×")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ks = [r[0] for r in results]
errors = [r[1] for r in results]
compressions = [r[2] for r in results]

ax1.plot(ks, errors, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Rank k')
ax1.set_ylabel('Relative Error')
ax1.set_title('Approximation Error vs Rank')
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(S)), S, color='steelblue')
ax2.set_xlabel('Index')
ax2.set_ylabel('Singular Value')
ax2.set_title('Singular Value Spectrum')

plt.tight_layout()
plt.show()

---
## 8. Matrix Completion (Recommender Systems)

Recommender systems exploit **low-rank structure** in user-item rating matrices:

$$R \approx UV^T$$

where:
- $U$ = user latent factors (n_users × k)
- $V$ = item latent factors (n_items × k)
- $k$ = latent dimension (much smaller than matrix dimensions)

In [ ]:
print("="*60)
print("EXAMPLE 8: Low-Rank Matrix for Recommendations")
print("="*60)

np.random.seed(42)

# Simulate user-item ratings (low-rank structure)
n_users, n_items, k = 20, 15, 3

# True low-rank factors
U_true = np.random.randn(n_users, k)
V_true = np.random.randn(n_items, k)

# True rating matrix (exactly low-rank)
R_true = U_true @ V_true.T

print(f"Rating matrix: {R_true.shape} ({n_users} users × {n_items} items)")
print(f"True underlying rank: {k}")
print(f"Verified rank: {np.linalg.matrix_rank(R_true)}")

# Add noise (simulating real ratings)
noise_level = 0.5
R_noisy = R_true + noise_level * np.random.randn(n_users, n_items)
print(f"\nNoisy matrix rank: {np.linalg.matrix_rank(R_noisy)}")

# Recover low-rank structure via SVD
U, S, Vt = np.linalg.svd(R_noisy, full_matrices=False)

print(f"\nSingular values (top 6): {np.round(S[:6], 2)}")
print("Note: First few singular values are dominant!")

# Rank-k approximation (recover the low-rank structure)
R_approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

recovery_error = np.linalg.norm(R_approx - R_true, 'fro') / np.linalg.norm(R_true, 'fro')
print(f"\nRecovery error vs true matrix: {recovery_error:.4f}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im0 = axes[0].imshow(R_true, cmap='coolwarm', aspect='auto')
axes[0].set_title('True Ratings (Low-Rank)')
axes[0].set_xlabel('Items')
axes[0].set_ylabel('Users')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(R_noisy, cmap='coolwarm', aspect='auto')
axes[1].set_title('Noisy Observations')
axes[1].set_xlabel('Items')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(R_approx, cmap='coolwarm', aspect='auto')
axes[2].set_title(f'Rank-{k} Approximation')
axes[2].set_xlabel('Items')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

---
## 9. Numerical Rank Issues

In practice, small singular values make exact rank computation problematic.

NumPy uses a **tolerance-based rank**:
$$\text{numerical rank} = \#\{\sigma_i > \epsilon \cdot \sigma_{\max}\}$$

In [ ]:
print("="*60)
print("EXAMPLE 9: Numerical Rank Issues")
print("="*60)

# Theoretically rank-deficient matrix
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

print(f"Matrix A:\n{A}")
print(f"\nTheoretical rank: 2")
print(f"(Row 3 = 2×Row 2 - Row 1)")

# Check with SVD
_, S, _ = np.linalg.svd(A)
print(f"\nSingular values: {S}")
print(f"Computed rank (default): {np.linalg.matrix_rank(A)}")

# Add tiny perturbation
print("\n" + "-"*40)
print("After tiny perturbation (1e-10 noise):")

np.random.seed(42)
epsilon = 1e-10
A_perturbed = A + epsilon * np.random.randn(3, 3)
_, S_perturbed, _ = np.linalg.svd(A_perturbed)

print(f"Perturbed singular values: {S_perturbed}")
print(f"Computed rank (default tol): {np.linalg.matrix_rank(A_perturbed)}")
print(f"Computed rank (tol=1e-8): {np.linalg.matrix_rank(A_perturbed, tol=1e-8)}")

# Condition number
cond = S[0] / S[1]  # Use second smallest since last is ~0
cond_full = S[0] / max(S[-1], 1e-16)
print(f"\nCondition number: {cond_full:.2e}")
print("Very large condition number indicates near rank-deficiency!")

# Visualization
plt.figure(figsize=(10, 4))

plt.subplot(121)
plt.bar(['σ₁', 'σ₂', 'σ₃'], S, color='steelblue')
plt.yscale('log')
plt.ylabel('Singular Value (log scale)')
plt.title('Original Matrix (Rank-Deficient)')
plt.axhline(1e-10, color='r', linestyle='--', label='Zero threshold')
plt.legend()

plt.subplot(122)
plt.bar(['σ₁', 'σ₂', 'σ₃'], S_perturbed, color='green')
plt.yscale('log')
plt.ylabel('Singular Value (log scale)')
plt.title('Perturbed Matrix')
plt.axhline(1e-8, color='r', linestyle='--', label='tol=1e-8')
plt.legend()

plt.tight_layout()
plt.show()

---
## 10. Rank Under Various Operations

Key properties:
- $\text{rank}(AB) \leq \min(\text{rank}(A), \text{rank}(B))$
- $\text{rank}(A^T) = \text{rank}(A)$
- $\text{rank}(A^TA) = \text{rank}(AA^T) = \text{rank}(A)$
- $\text{rank}(uv^T) = 1$ for non-zero vectors

In [ ]:
print("="*60)
print("EXAMPLE 10: Rank Under Operations")
print("="*60)

np.random.seed(42)

A = np.random.randn(4, 3)
B = np.random.randn(3, 5)

rank_A = np.linalg.matrix_rank(A)
rank_B = np.linalg.matrix_rank(B)

print(f"A: {A.shape}, rank = {rank_A}")
print(f"B: {B.shape}, rank = {rank_B}")

# Product
print("\n" + "-"*40)
print("MATRIX PRODUCT:")
AB = A @ B
rank_AB = np.linalg.matrix_rank(AB)
print(f"AB: {AB.shape}, rank = {rank_AB}")
print(f"rank(AB) ≤ min(rank(A), rank(B)) = min({rank_A}, {rank_B}) = {min(rank_A, rank_B)} ✓")

# Transpose
print("\n" + "-"*40)
print("TRANSPOSE:")
print(f"rank(A) = {rank_A}")
print(f"rank(Aᵀ) = {np.linalg.matrix_rank(A.T)}")
print("rank(A) = rank(Aᵀ) ✓")

# Gram matrix
print("\n" + "-"*40)
print("GRAM MATRIX:")
print(f"rank(A) = {rank_A}")
print(f"rank(AᵀA) = {np.linalg.matrix_rank(A.T @ A)}")
print(f"rank(AAᵀ) = {np.linalg.matrix_rank(A @ A.T)}")
print("All equal to rank(A) ✓")

# Outer product
print("\n" + "-"*40)
print("OUTER PRODUCT:")
u = np.random.randn(4)
v = np.random.randn(3)
outer = np.outer(u, v)
print(f"Outer product uvᵀ: {outer.shape}")
print(f"rank(uvᵀ) = {np.linalg.matrix_rank(outer)}")
print("Outer products always have rank 1! ✓")

---
## 11. Visualization: Rank and Geometry

Rank determines the dimensionality of the output (image) of a linear transformation.

In [ ]:
print("="*60)
print("VISUALIZATION: Rank and Geometry")
print("="*60)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Unit square vertices
square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]]).T

# Rank 2 (full rank)
A1 = np.array([[1, 0.5], [0, 1]])
transformed1 = A1 @ square

axes[0].fill(square[0], square[1], alpha=0.3, color='blue', label='Original')
axes[0].fill(transformed1[0], transformed1[1], alpha=0.3, color='red', label='Transformed')
axes[0].set_xlim(-0.5, 2)
axes[0].set_ylim(-0.5, 2)
axes[0].set_aspect('equal')
axes[0].set_title(f'Rank 2 (Full Rank)\nPreserves area')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rank 1 (collapses to line)
A2 = np.array([[1, 0.5], [2, 1]])  # Columns are dependent (col2 = 0.5*col1 + ...)
transformed2 = A2 @ square

axes[1].fill(square[0], square[1], alpha=0.3, color='blue', label='Original')
axes[1].plot(transformed2[0], transformed2[1], 'r-', linewidth=3, label='Transformed (line)')
axes[1].set_xlim(-0.5, 3)
axes[1].set_ylim(-0.5, 4)
axes[1].set_aspect('equal')
axes[1].set_title(f'Rank 1\nCollapses to a line')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Rank 0 (collapses to point)
A3 = np.zeros((2, 2))
transformed3 = A3 @ square

axes[2].fill(square[0], square[1], alpha=0.3, color='blue', label='Original')
axes[2].plot(0, 0, 'ro', markersize=15, label='Transformed (point)')
axes[2].set_xlim(-0.5, 2)
axes[2].set_ylim(-0.5, 2)
axes[2].set_aspect('equal')
axes[2].set_title(f'Rank 0\nCollapses to origin')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nGeometric Interpretation:")
print("- Rank 2: Square → Parallelogram (preserves 2D structure)")
print("- Rank 1: Square → Line (loses 1 dimension)")
print("- Rank 0: Square → Point (loses all dimensions)")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Definition** | Max # of linearly independent rows/columns |
| **Computation** | Count non-zero singular values |
| **Rank-Nullity** | rank(A) + nullity(A) = n |
| **Upper bound** | rank(A) ≤ min(m, n) |
| **Product bound** | rank(AB) ≤ min(rank(A), rank(B)) |
| **Transpose** | rank(A) = rank(Aᵀ) |
| **ML significance** | Detects feature redundancy, enables compression |